# CIDER Dataset Quick Start

This notebook will walk you through the CIDER dataset to

1. Explore the original data tables, disclosure variants, and study materials.
2. Run basic preference analysis across users, scenarios, and variants.
3. (Optional) Generate new disclosure variants from a natural-language description or a PrivacyLens seed.


## 0. Set-Up

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dataset" / "original").is_dir()
    and (path / "evaluation" / "helper" / "constants.py").is_file()
)
sys.path.insert(0, str(ROOT / "evaluation" / "helper"))

from constants import (
    ORIGINAL_DIR,
    RATINGS_PATH,
    SCENARIOS_PATH,
    VARIANT_LABEL_MAP,
    VARIANT_SUFFIXES,
    VISUAL_CARDS_DIR,
)

assert RATINGS_PATH.is_file() and SCENARIOS_PATH.is_file()
ORIGINAL_DIR

## 1. Explore CIDER dataset

Load the two tables and inspect shapes / a few rows.

In [ ]:
from load_data import read_table

ratings = read_table(RATINGS_PATH)
scenarios = read_table(SCENARIOS_PATH)

print(f"ratings:   {ratings.shape[0]} rows × {ratings.shape[1]} cols")
print(f"scenarios: {scenarios.shape[0]} rows × {scenarios.shape[1]} cols")
print(f"users:     {ratings['Participant_ID'].nunique()}")
print(f"scenario_ids in scenarios: {scenarios['scenario_id'].nunique()}")
print(f"visual cards: {len(list(VISUAL_CARDS_DIR.glob('*.png')))}")
print(f"loaded from: {RATINGS_PATH.name}, {SCENARIOS_PATH.name}")

display(Markdown("### Ratings (head)"))
display(ratings.head(3))
display(Markdown("### Scenarios (selected columns)"))
scenario_preview_cols = [
    c
    for c in [
        "scenario_id",
        "sender_format",
        "subject_format",
        "recipient_format",
        "transmission_principle",
        "3rd_narrative",
    ]
    if c in scenarios.columns
]
display(scenarios[scenario_preview_cols].head(3))

In [ ]:
# Peek at one scenario's 9 disclosure variants
sid = scenarios["scenario_id"].iloc[0]
row = scenarios.loc[scenarios["scenario_id"] == sid].iloc[0]

display(Markdown(f"### Scenario `{sid}`"))
print(row.get("3rd_narrative", ""))
print()
for suffix in VARIANT_SUFFIXES:
    col = f"variant_{suffix}_cleaned"
    if col not in row.index:
        col = f"variant_{suffix}"
    label = VARIANT_LABEL_MAP[suffix]
    text = row[col] if col in row.index else ""
    print(f"[{label} / var{suffix}] {text}")

# Show role visual cards if present
for role in ("data_sender", "data_subject", "data_recipient"):
    png = VISUAL_CARDS_DIR / f"{sid}_{role}.png"
    if png.is_file():
        display(Markdown(f"**{role}** (`{png.name}`)") )
        display(Image(filename=str(png), width=280))

## 2. Dataset analysis

In [ ]:
from load_data import rating_to_yes_no

role_col = "Role" if "Role" in ratings.columns else "role"
cond_col = "Condition" if "Condition" in ratings.columns else "condition"

display(Markdown("### Role × condition counts"))
display(pd.crosstab(ratings[role_col], ratings[cond_col], margins=True))

# Per-user scenario slots: s0_id … s9_id
slot_cols = [c for c in ratings.columns if c.startswith("s") and c.endswith("_id")]
n_slots = ratings[slot_cols].notna().sum(axis=1)
display(Markdown("### Scenarios rated per user"))
display(n_slots.value_counts().sort_index().rename("n_users").to_frame())


def to_yes_no(series: pd.Series) -> pd.Series:
    """Map ratings via ``rating_to_yes_no``; skip unrecognized values."""
    out = []
    for value in series.dropna():
        try:
            out.append(rating_to_yes_no(value))
        except ValueError:
            continue
    return pd.Series(out, dtype=str)


# Flatten variant ratings only for slots with a non-null scenario id.
flat_parts = []
for si in range(10):
    id_col = f"s{si}_id"
    if id_col not in ratings.columns:
        continue
    mask = ratings[id_col].notna()
    for suffix in VARIANT_SUFFIXES:
        col = f"s{si}_var{suffix}_rating"
        if col not in ratings.columns:
            continue
        flat_parts.append(to_yes_no(ratings.loc[mask, col]))

flat = pd.concat(flat_parts, ignore_index=True) if flat_parts else pd.Series(dtype=str)
display(Markdown("### Overall Yes/No (slots with non-null scenario id only)"))
display(flat.value_counts(normalize=True).rename("share").to_frame())
print(f"Overall Yes rate: {(flat == 'YES').mean():.2%}  (n={len(flat)})")

rows = []
for suffix in VARIANT_SUFFIXES:
    parts = []
    for si in range(10):
        id_col = f"s{si}_id"
        col = f"s{si}_var{suffix}_rating"
        if id_col not in ratings.columns or col not in ratings.columns:
            continue
        mask = ratings[id_col].notna()
        parts.append(to_yes_no(ratings.loc[mask, col]))
    vals = pd.concat(parts, ignore_index=True) if parts else pd.Series(dtype=str)
    yes = (vals == "YES").mean() if len(vals) else float("nan")
    rows.append(
        {
            "variant": suffix,
            "label": VARIANT_LABEL_MAP[suffix],
            "n": int(len(vals)),
            "yes_rate": float(yes),
        }
    )
variant_yes = pd.DataFrame(rows)
display(Markdown("### Yes-rate by disclosure variant (G×I)"))
display(variant_yes)

In [ ]:
sys.path.insert(0, str(ROOT / "dataset"))
from explore import plot_variant_level_yes_rates

heatmap_path = plot_variant_level_yes_rates(ratings)
display(Markdown("### Average Yes rate by variant (G×I), role × condition"))
display(Image(filename=str(heatmap_path), width=780))
print(f"Saved → {heatmap_path}")


In [ ]:
# Inter-user agreement: mean pairwise match rate over complete 9-variant boundaries
sys.path.insert(0, str(ROOT / "dataset"))
from explore import plot_inter_user_agreement

agreement_paths = plot_inter_user_agreement(ratings)
display(Markdown("### Inter-user agreement"))
for path in agreement_paths:
    display(Markdown(f"**{path.stem}**"))
    # Fixed display height; width scales with each role's figure.
    display(Image(filename=str(path), height=280))


## 3. Create new scenarios / contexts

### 3.0 Setup for generation

In [ ]:
VG_DIR = ROOT / "dataset" / "extensibility"
sys.path.insert(0, str(VG_DIR))

from script import (
    generate_from_description,
    generate_from_privacylens_row,
    normalize_description,
    privacy_lens_row_to_description,
    resolve_chat_credentials,
)

# Load repo ``.env`` and pick OpenAI or OpenRouter credentials.
API_KEY, BASE_URL, MODEL = resolve_chat_credentials(env_file=ROOT / ".env")
print(f"Generation ready: model={MODEL} base_url={BASE_URL}")

### 3.1 Choice A: natural-language description

Describe the scenario directly using the eight normalized fields.

In [ ]:
choice_a = {
    "name": "example-adapted-from-main8",
    "data_type": "detailed future travel plans including departure and return dates",
    "sensitive_info_items": [
        "John is planning an upcoming solo photography trip to Patagonia.",
        "John plans to depart for Patagonia on September 22, 2022.",
        "John plans to return from Patagonia on October 6, 2022.",
        "The trip is planned around photographing Patagonia's autumn colors.",
    ],
    "data_sender": "John",
    "data_subject": "John",
    "data_recipient": "Emily (a casual acquaintance from John's photography club)",
    "transmission_principle": "an email",
    "story": (
        "A person shares detailed plans for an upcoming solo trip, including departure and return dates with a casual acquaintance by sending an email."
    ),
}

normalize_description(choice_a)

In [ ]:
result_a = generate_from_description(
    choice_a,
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
)
display(Markdown(result_a["output"]))

### 3.2 Choice B: PrivacyLens seed

An example `main404` from [PrivacyLens](https://github.com/salt-nlp/privacylens).

In [ ]:
privacy_lens_seed = {
    "name": "main404",
    "data_type": "work history",
    "data_subject": "a famous client",
    "data_sender": "music therapist",
    "data_sender_name": "Jane",
    "data_recipient": "social media followers",
    "transmission_principle": "send a facebook post",
    "data_type_concrete": "Detailed notes on sessions with a famous client, including their extensive and confidential work history in the music industry.",
    "data_subject_concrete": "A famous client",
    "data_sender_concrete": "Jane",
    "data_recipient_concrete": "Social media followers",
    "data_sender_name": "Jane",
    "data_subject_name": "David Melrose",
    "data_recipient_name": "Social media followers",
    "sensitive_info": [
        "David Melrose is a famous pop star.",
        "David has struggled with the pressures of public performances.",
        "Our sessions have been focused on developing personalized strategies to enhance his stage presence and confidence.",
        "Since beginning therapy, there has been a measurable improvement in his ability to perform under pressure.",
        "David has reported a higher level of satisfaction with his musical engagements."
    ],
}

privacy_lens_story = (
    "A music therapist shares a famous client's personal therapy experience with social-media followers in a Facebook post."
)

privacy_lens_row_to_description(
    privacy_lens_seed,
    story=privacy_lens_story,
)

In [ ]:
result_b = generate_from_privacylens_row(
    privacy_lens_seed,
    story=privacy_lens_story,
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
)
display(Markdown(result_b["output"]))